# Chronogeometric Valuation Demo: S&P 500 & Bitcoin Temporal Resonance
Based on *The Chronal Ruler* (Siddiquie, 2026)
Harmonic cadences: P = [3, 6, 9, 12] years
Chronal anchor: 2008-09-15 (Lehman collapse) → τ₀ = 2008.71

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from datetime import datetime, timedelta
from scipy.signal import hilbert
from statsmodels.tsa.seasonal import STL

# Load data (simulated for demo; replace with yfinance/kraken API in prod)
np.random.seed(42)
dates = pd.date_range("2010-01-01", "2026-05-11", freq="D")
n = len(dates)

# Simulate S&P 500 log-return (realistic volatility clustering)
sp500 = np.cumsum(np.random.normal(0.0003, 0.01, n)) + 1000
sp500 += 5 * np.sin(2*np.pi * np.arange(n)/365.25) # annual seasonality

# Simulate Bitcoin price (log-price with bull/bear cycles)
btc_log = np.cumsum(np.random.normal(0.001, 0.04, n)) + 5
btc_log += 2 * np.sin(2*np.pi * np.arange(n)/(3*365.25)) # ~3-yr cycle
btc_log += 1.5 * np.sin(2*np.pi * np.arange(n)/(9*365.25)) # ~9-yr cycle

# Chronal anchor: 2008-09-15 → t₀ in years since epoch
t0 = (datetime(2008,9,15) - datetime(1970,1,1)).days / 365.25
t = (dates - datetime(1970,1,1)).days / 365.25

# Harmonic periods (years)
P = np.array([3.0, 6.0, 9.0, 12.0])
theta = 2*np.pi * ((t - t0)[:, None] % P) / P # shape (n, 4)

# Compute resonance metric between SPX and BTC at each day
# Using analytic signal (Hilbert transform) to extract instantaneous phase
sp_phase = np.angle(hilbert(sp500))
btc_phase = np.angle(hilbert(btc_log))

# Normalize to [0, 2π)
sp_phase = (sp_phase + 2*np.pi) % (2*np.pi)
btc_phase = (btc_phase + 2*np.pi) % (2*np.pi)

# Resonance strength: cos²(Δθ/2)
delta_theta = np.abs(sp_phase - btc_phase)
R = np.cos(delta_theta / 2)**2

## Visualization: Temporal Resonance Over Time

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dates, y=R,
    mode='lines',
    name='Resonance Strength 𝓡(t)',
    line=dict(color='darkblue', width=2),
    hovertemplate='Date: %{x}<br>Resonance: %{y:.3f}<extra></extra>'
))
fig.add_hline(y=0.85, line_width=1, line_dash="dash", line_color="red",
              annotation_text="High Resonance Threshold (𝓡 > 0.85)",
              annotation_position="top right")
fig.update_layout(
    title="Temporal Resonance Between S&P 500 and Bitcoin<br><sup>Using Chronal Ruler Harmonic Cadences (P=3,6,9,12 yr)</sup>",
    xaxis_title="Date",
    yaxis_title="Resonance Strength 𝓡 ∈ [0,1]",
    height=500,
    template="plotly_white"
)
fig.show()

## Phase Portrait: 3D Harmonic Lattice (P=3,6,9)
Points where 𝓡 > 0.85 are highlighted — they lie near diagonal in phase space.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
mask = R > 0.85
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

# Plot full trajectory
ax.plot(theta[:,0], theta[:,1], theta[:,2], color='lightgray', alpha=0.3, linewidth=0.5)

# Highlight resonant points
ax.scatter(theta[mask,0], theta[mask,1], theta[mask,2],
           c='crimson', s=30, alpha=1.0, label='High Resonance Events')

ax.set_xlabel(r'$\theta_3$ (3-yr)')
ax.set_ylabel(r'$\theta_6$ (6-yr)')
ax.set_zlabel(r'$\theta_9$ (9-yr)')
ax.set_title('Chronal Lattice: Resonance Clusters in Phase Space')
ax.legend()
plt.tight_layout()
plt.show()